In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)
]

columns = ["order_id","customer_id","product","category","city","date","amount","quantity"]

df = spark.createDataFrame(data, columns)
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df.write.format("delta").save("/Volumes/medallion_arch/bronze/raw_data")

In [0]:
raw_df=spark.read.format("delta").load("/Volumes/medallion_arch/bronze/raw_data")

In [0]:
silver_data=spark.read.format("delta").load("/Volumes/medallion_arch/bronze/raw_data")
silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
silver_data.fillna({
    "city":"Unknown",
     "amount":0
}).display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
from pyspark.sql.functions import col,when
silver_data=silver_data.withColumn('Data Validation',
                 when(col("amount")<0,"Invalid").otherwise("Valid"))
silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity,Data Validation
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1,Valid
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2,Valid
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1,Valid
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1,Valid
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1,Valid
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1,Valid
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1,Valid
107,C005,Headphones,Accessories,null,2024-01-08,3000,1,Valid
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1,Invalid
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,Valid


In [0]:
silver_data=silver_data.filter(col("amount")>=0)
silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity,Data Validation
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1,Valid
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1,Valid
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1,Valid
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1,Valid
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1,Valid
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1,Valid
107,C005,Headphones,Accessories,null,2024-01-08,3000,1,Valid
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,Valid
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,Valid


In [0]:
silver_data=silver_data.fillna({
    "city":"Unknown"
})
silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity,Data Validation
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1,Valid
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1,Valid
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1,Valid
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1,Valid
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1,Valid
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1,Valid
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1,Valid
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,Valid
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,Valid


In [0]:
from pyspark.sql.functions import trim,col
silver_data=silver_data.withColumn("order_id",trim(col("order_id")))
silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity,Data Validation
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1,Valid
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1,Valid
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1,Valid
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1,Valid
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1,Valid
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1,Valid
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1,Valid
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,Valid
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,Valid


In [0]:
silver_data=silver_data.dropDuplicates()
silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity,Data Validation
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1,Valid
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1,Valid
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1,Valid
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1,Valid
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1,Valid
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,Valid
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1,Valid
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1,Valid


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = Window.partitionBy("order_id").orderBy(col("date").desc())

silver_data = silver_data.withColumn("rn", row_number().over(window))

silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity,Data Validation,rn
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1,Valid,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1,Valid,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1,Valid,2
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1,Valid,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1,Valid,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1,Valid,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1,Valid,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,Valid,1


In [0]:
silver_data=silver_data.filter(col("rn")!=2)
silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity,Data Validation,rn
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1,Valid,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1,Valid,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1,Valid,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1,Valid,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1,Valid,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1,Valid,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,Valid,1


In [0]:
silver_data=silver_data.withColumn("amount",col("amount").cast("int"))

In [0]:
silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity,Data Validation,rn
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1,Valid,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1,Valid,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1,Valid,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1,Valid,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1,Valid,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1,Valid,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,Valid,1


In [0]:
silver_data=silver_data.drop("Data Validation","rn")

In [0]:
silver_data.write.format("delta").mode("overwrite").save("/Volumes/medallion_arch/gold/cleaned-data")

In [0]:
gold_data=spark.read.format("delta").load("/Volumes/medallion_arch/gold/cleaned-data")

In [0]:
gold_data.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
from pyspark.sql.functions import sum,col
gold_data.groupBy("product").agg(
    sum(col("amount")).alias("Sales")
).display()

product,Sales
Tablet,22000
Mobile,33000
Watch,8000
Laptop,105000
Headphones,3000


In [0]:
gold_data.groupBy("category").agg(
    sum(col("amount")).alias("Category")
).display()

category,Category
Electronics,160000
Accessories,11000


In [0]:
gold_data.groupBy("city").agg(
    sum(col("amount")).alias("City")
).display()

city,City
Delhi,55000
Chennai,33000
Unknown,3000
Hyderabad,72000
Mumbai,8000


In [0]:
from pyspark.sql.functions import count,col
gold_data.groupBy("customer_id").agg(
    count(col("order_id")).alias("Orders")
).display()

customer_id,Orders
C005,1
C004,1
C003,1
C001,2
C002,1
C007,1


In [0]:
gold_data.groupBy("customer_id") \
  .agg(sum("amount").alias("total_spent")) \
  .orderBy(col("total_spent").desc()) \
  .display()

customer_id,total_spent
C001,72000
C003,55000
C002,18000
C007,15000
C004,8000
C005,3000


In [0]:
gold_data=gold_data.withColumn("revenue",col("amount")*col("quantity"))
gold_data.display()

order_id,customer_id,product,category,city,date,amount,quantity,revenue
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1,50000
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1,22000
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1,55000
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1,18000
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1,8000
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1,3000
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2,30000


In [0]:
gold_data.groupBy("product").agg(sum("amount").alias("sales")).orderBy(col("sales").desc()).limit(1).display()

product,sales
Laptop,105000


In [0]:
gold_data.select("product","revenue").orderBy(col("revenue").desc()).limit(1).display()

product,revenue
Laptop,55000
